# TF-IDF 文本向量化

机器学习无法直接处理文本内容，需要将文本内容转化为向量

相关理论见[词嵌入](../docs/002_word_embedding.md)

In [1]:
# 加载清理好的数据
import pandas as pd

from sentiment_analysis.paths import CLEANED_DIR

train_data = pd.read_csv(CLEANED_DIR / "train.csv")
validation_data = pd.read_csv(CLEANED_DIR / "val.csv")

X_train = train_data["text"]
X_validation = validation_data["text"]

print(f"Train: {len(X_train):,}")
print(f"Validation: {len(X_validation):,}")

Train: 22,413
Validation: 2,491


In [2]:
# 构建 TF-IDF 特征矩阵
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,         # 将文本统一转化为小写
    ngram_range=(1, 1),     # 仅使用 Unigram
    min_df=2,               # 词项至少出现在两条训练评论中
    max_features=20_000,    # 最多保留 20,000 个词项特征
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_validation_tfidf = vectorizer.transform(X_validation)

print("Train shape:", X_train_tfidf.shape)
print("Validation shape:", X_validation_tfidf.shape)
print("Vocabulary size:", len(vectorizer.vocabulary_))

Train shape: (22413, 20000)
Validation shape: (2491, 20000)
Vocabulary size: 20000


In [3]:
n_samples, n_features = X_train_tfidf.shape

nonzero = X_train_tfidf.nnz
density = nonzero / (n_samples * n_features)

print(f"Samples: {n_samples:,}")
print(f"Features: {n_features:,}")
print(f"Non-zero values: {nonzero:,}")
print(f"Density: {density:.4%}")
print(f"Sparsity: {1 - density:.4%}")

Samples: 22,413
Features: 20,000
Non-zero values: 2,964,013
Density: 0.6612%
Sparsity: 99.3388%


In [4]:
from sklearn.feature_extraction.text import CountVectorizer

counter = CountVectorizer(
    lowercase=True,
    ngram_range=(1, 1),
    min_df=2,
)

X_counts = counter.fit_transform(X_train)

terms = counter.get_feature_names_out()

term_frequency = X_counts.sum(axis=0).A1
document_frequency = X_counts.getnnz(axis=0)

term_stats = pd.DataFrame({
    "term": terms,
    "tf": term_frequency,
    "df": document_frequency,
})

display(term_stats.sort_values("tf", ascending=False).head(20))

,term,tf,df
38163,the,302003,22229
1754,and,146963,21661
26573,of,130616,21285
38602,to,121825,21062
20132,is,96101,20094
20188,it,86644,19968
19083,in,84313,19743
38288,this,68072,20287
38155,that,65804,18225
41440,was,43238,14528


In [5]:
# Bigram TF-IDF
bigram_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(2, 2),
    min_df=2,
    max_features=20_000,
)

X_train_bigram = bigram_vectorizer.fit_transform(X_train)
X_validation_bigram = bigram_vectorizer.transform(X_validation)

print("Train shape:", X_train_bigram.shape)
print("Validation shape:", X_validation_bigram.shape)

print("Sample features:")
print(bigram_vectorizer.get_feature_names_out()[:20])

Train shape: (22413, 20000)
Validation shape: (2491, 20000)
Sample features:
['000 000' '10 10' '10 and' '10 because' '10 but' '10 for' '10 from'
 '10 is' '10 it' '10 lines' '10 minutes' '10 or' '10 out' '10 stars'
 '10 the' '10 year' '10 years' '100 years' '12 year' '12 years']


In [6]:
# Unigram + Bigram TF-IDF
combined_vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
    max_features=20_000,
)

X_train_combined = combined_vectorizer.fit_transform(X_train)
X_validation_combined = combined_vectorizer.transform(X_validation)

print("Train shape:", X_train_combined.shape)
print("Validation shape:", X_validation_combined.shape)

print("Sample features:")
print(combined_vectorizer.get_feature_names_out()[:20])

Train shape: (22413, 20000)
Validation shape: (2491, 20000)
Sample features:
['00' '000' '10' '10 10' '10 and' '10 because' '10 but' '10 for' '10 is'
 '10 it' '10 lines' '10 minutes' '10 out' '10 stars' '10 the' '10 year'
 '10 years' '100' '1000' '11']


In [7]:
feature_sets = {
    "Unigram": X_train_tfidf,
    "Bigram": X_train_bigram,
    "Unigram + Bigram": X_train_combined,
}

results = []

for name, matrix in feature_sets.items():
    n_samples, n_features = matrix.shape

    results.append({
        "REPRESENTATION": name,
        "FEATURES": n_features,
        "AVERAGE ACTIVE FEATURES": matrix.nnz / n_samples,
        "DENSITY (%)": matrix.nnz / (n_samples * n_features) * 100,
    })

feature_summary = pd.DataFrame(results)

display(feature_summary.round(3))

,Representation,Features,Average Active Features,Density (%),Memory (MB)
0,Unigram,20000,132.245,0.661,34.006
1,Bigram,20000,98.155,0.491,25.262
2,Unigram + Bigram,20000,210.431,1.052,54.060


In [8]:
combined_terms = combined_vectorizer.get_feature_names_out()

unigram_count = sum(" " not in term for term in combined_terms)
bigram_count = sum(" " in term for term in combined_terms)

print("Unigram features:", unigram_count)
print("Bigram features:", bigram_count)
print("Total features:", len(combined_terms))

Unigram features: 7564
Bigram features: 12436
Total features: 20000
